<a href="https://colab.research.google.com/github/gnoejh/AIBookGitHub/blob/main/12_grid_world_q.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grid World Reinforcement Learning - Temporal Difference Learning

This notebook demonstrates **Temporal Difference (TD) Learning** algorithms applied to our grid world problem. Students will learn:

1. **Q-Learning**: Off-policy learning that learns the optimal action-value function
2. **SARSA**: On-policy learning that learns the action-value function for the current policy
3. **Exploration vs Exploitation**: Using epsilon-greedy strategies
4. **Learning from Experience**: No model required, learning from trial and error
5. **Algorithm Comparison**: Comparing Q-Learning vs SARSA behavior

## Learning Objectives

- Understand temporal difference learning concepts
- Learn the difference between on-policy and off-policy methods
- Visualize Q-table evolution during learning
- Compare exploration strategies and their effects
- See how algorithms learn from experience rather than complete environment knowledge

Let's start by setting up our environment and TD learning algorithms!

In [48]:
# Import required libraries
import numpy as np
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import time
from enum import Enum
from typing import Tuple, List, Optional, Dict
import random

print("Libraries imported successfully!")
print("Ready to create our TD Learning algorithms.")
print("Note: Using text-based visualization for better compatibility.")

Libraries imported successfully!
Ready to create our TD Learning algorithms.
Note: Using text-based visualization for better compatibility.


## 1. Environment Setup (Reused from DP)

We'll reuse our GridWorld environment from the Dynamic Programming notebook, but now we'll use it for **experience-based learning** rather than model-based planning.

In [49]:
# Define the cell types and actions (same as before)
class CellType(Enum):
    EMPTY = 0
    OBSTACLE = 1
    GOAL = 2
    PENALTY = 3
    AGENT = 4

class Action(Enum):
    UP = 0
    DOWN = 1
    LEFT = 2
    RIGHT = 3

# GridWorld Environment Class (reused)
class GridWorld:
    def __init__(self, width=5, height=5):
        self.width = width
        self.height = height
        self.grid = np.zeros((height, width), dtype=int)
        self.agent_pos = [0, 0]  # [row, col]
        self.goal_pos = [height-1, width-1]
        self.start_pos = [0, 0]
        
        # Rewards
        self.step_reward = -0.1  # Small penalty for each step
        self.goal_reward = 10.0  # Large reward for reaching goal
        self.obstacle_penalty = -1.0  # Penalty for hitting obstacle
        self.penalty_reward = -5.0  # Large penalty for penalty cells
        
        # Initialize grid
        self._setup_default_grid()
        
    def _setup_default_grid(self):
        """Setup a default grid with some obstacles and penalties"""
        # Clear the grid
        self.grid.fill(CellType.EMPTY.value)
        
        # Add some obstacles
        if self.width >= 5 and self.height >= 5:
            self.grid[1, 2] = CellType.OBSTACLE.value
            self.grid[2, 2] = CellType.OBSTACLE.value
            self.grid[3, 1] = CellType.OBSTACLE.value
            
            # Add a penalty cell
            self.grid[2, 3] = CellType.PENALTY.value
        
        # Set goal
        self.grid[self.goal_pos[0], self.goal_pos[1]] = CellType.GOAL.value
        
    def reset(self):
        """Reset the agent to starting position"""
        self.agent_pos = self.start_pos.copy()
        return self.get_state()
    
    def get_state(self):
        """Get current state as tuple (row, col)"""
        return tuple(self.agent_pos)
    
    def is_valid_action(self, action):
        """Check if action is valid from current position"""
        new_pos = self._get_new_position(action)
        return self._is_valid_position(new_pos)
    
    def _get_new_position(self, action):
        """Calculate new position after taking action"""
        row, col = self.agent_pos
        
        if action == Action.UP:
            return [row - 1, col]
        elif action == Action.DOWN:
            return [row + 1, col]
        elif action == Action.LEFT:
            return [row, col - 1]
        elif action == Action.RIGHT:
            return [row, col + 1]
        else:
            return [row, col]  # Invalid action, stay in place
    
    def _is_valid_position(self, pos):
        """Check if position is within bounds and not an obstacle"""
        row, col = pos
        
        # Check bounds
        if row < 0 or row >= self.height or col < 0 or col >= self.width:
            return False
        
        # Check if it's an obstacle
        if self.grid[row, col] == CellType.OBSTACLE.value:
            return False
        
        return True
    
    def step(self, action):
        """Take a step in the environment"""
        # Calculate new position
        new_pos = self._get_new_position(action)
        
        # Check if move is valid
        if not self._is_valid_position(new_pos):
            # Invalid move - stay in place and get penalty
            reward = self.obstacle_penalty
            done = False
        else:
            # Valid move - update position
            self.agent_pos = new_pos
            
            # Calculate reward based on cell type
            cell_type = self.grid[new_pos[0], new_pos[1]]
            
            if cell_type == CellType.GOAL.value:
                reward = self.goal_reward
                done = True
            elif cell_type == CellType.PENALTY.value:
                reward = self.penalty_reward
                done = False
            else:  # Empty cell
                reward = self.step_reward
                done = False
        
        # Return state, reward, done, info
        return self.get_state(), reward, done, {}
    
    def get_possible_actions(self):
        """Get all possible actions from current state"""
        possible_actions = []
        for action in Action:
            if self.is_valid_action(action):
                possible_actions.append(action)
        return possible_actions

print("GridWorld environment created successfully!")
print("✓ Same environment as DP, but now for experience-based learning")
print("✓ No model knowledge required for TD learning")

GridWorld environment created successfully!
✓ Same environment as DP, but now for experience-based learning
✓ No model knowledge required for TD learning


## 2. Visualization System with Q-Table Display

We'll enhance our visualization to show Q-values for each state-action pair.

In [50]:
class QTableVisualizer:
    def __init__(self, grid_world):
        self.grid_world = grid_world
        
        # Define symbols for different cell types
        self.symbols = {
            CellType.EMPTY.value: '⬜',
            CellType.OBSTACLE.value: '🚫',
            CellType.GOAL.value: '🎯',
            CellType.PENALTY.value: '🔥'
        }
        
        self.agent_symbol = '🤖'
        self.action_symbols = {
            Action.UP: '↑',
            Action.DOWN: '↓', 
            Action.LEFT: '←',
            Action.RIGHT: '→'
        }
        
    def create_grid_html(self, show_agent=True):
        """Create HTML representation of the grid"""
        html = "<div style='font-family: monospace; font-size: 24px; line-height: 1.2;'>"
        html += "<h3 style='text-align: center; margin: 10px 0;'>Grid World Environment</h3>"
        html += "<div style='display: inline-block; border: 2px solid #333; padding: 5px;'>"
        
        for i in range(self.grid_world.height):
            html += "<div style='display: flex;'>"
            for j in range(self.grid_world.width):
                # Check if agent is at this position
                if show_agent and [i, j] == self.grid_world.agent_pos:
                    symbol = self.agent_symbol
                    bg_color = '#E3F2FD'  # Light blue background for agent
                else:
                    cell_type = self.grid_world.grid[i, j]
                    symbol = self.symbols.get(cell_type, '⬜')
                    bg_color = '#F5F5F5'  # Light gray background
                
                html += f"<div style='width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border: 1px solid #ccc; background-color: {bg_color};'>{symbol}</div>"
            html += "</div>"
        
        html += "</div></div>"
        return html
    
    def create_q_table_html(self, q_table, title="Q-Table"):
        """Create HTML representation of Q-table values"""
        html = "<div style='font-family: monospace; font-size: 10px; line-height: 1.1;'>"
        html += f"<h4 style='text-align: center; margin: 10px 0;'>{title}</h4>"
        html += "<div style='display: inline-block; border: 2px solid #333; padding: 5px;'>"
        
        # Normalize Q-values for color coding
        max_q = np.max(q_table) if np.max(q_table) > 0 else 1
        min_q = np.min(q_table) if np.min(q_table) < 0 else -1
        
        for i in range(self.grid_world.height):
            html += "<div style='display: flex;'>"
            for j in range(self.grid_world.width):
                if self.grid_world.grid[i, j] == CellType.OBSTACLE.value:
                    # Show obstacle
                    html += "<div style='width: 80px; height: 60px; display: flex; align-items: center; justify-content: center; border: 1px solid #ccc; background-color: #000000; color: white;'>🚫</div>"
                else:
                    # Show Q-values for this state
                    html += "<div style='width: 80px; height: 60px; border: 1px solid #ccc; background-color: #f9f9f9; position: relative;'>"
                    
                    # Add Q-values for each action (arranged in cross pattern)
                    for action_idx, action in enumerate(Action):
                        q_val = q_table[i, j, action_idx]
                        
                        # Color based on Q-value
                        if q_val > 0:
                            intensity = min(q_val / max_q, 1.0)
                            color = f'rgba(0, 255, 0, {intensity})'
                        elif q_val < 0:
                            intensity = min(abs(q_val) / abs(min_q), 1.0)
                            color = f'rgba(255, 0, 0, {intensity})'
                        else:
                            color = 'rgba(128, 128, 128, 0.1)'
                        
                        # Position based on action
                        if action == Action.UP:
                            style = 'top: 0px; left: 25px; width: 30px; height: 15px;'
                        elif action == Action.DOWN:
                            style = 'bottom: 0px; left: 25px; width: 30px; height: 15px;'
                        elif action == Action.LEFT:
                            style = 'top: 22px; left: 0px; width: 25px; height: 16px;'
                        else:  # RIGHT
                            style = 'top: 22px; right: 0px; width: 25px; height: 16px;'
                        
                        html += f"<div style='position: absolute; {style} background-color: {color}; border: 1px solid #ddd; display: flex; align-items: center; justify-content: center; font-size: 8px; font-weight: bold;'>{q_val:.1f}</div>"
                    
                    html += "</div>"
            html += "</div>"
        
        html += "</div></div>"
        return html
    
    def create_policy_from_q_html(self, q_table, title="Policy from Q-Table"):
        """Create policy visualization from Q-table"""
        html = "<div style='font-family: monospace; font-size: 20px; line-height: 1.2;'>"
        html += f"<h4 style='text-align: center; margin: 10px 0;'>{title}</h4>"
        html += "<div style='display: inline-block; border: 2px solid #333; padding: 5px;'>"
        
        for i in range(self.grid_world.height):
            html += "<div style='display: flex;'>"
            for j in range(self.grid_world.width):
                if self.grid_world.grid[i, j] == CellType.OBSTACLE.value:
                    symbol = '🚫'
                    bg_color = '#000000'
                elif self.grid_world.grid[i, j] == CellType.GOAL.value:
                    symbol = '🎯'
                    bg_color = '#FFD700'
                elif self.grid_world.grid[i, j] == CellType.PENALTY.value:
                    symbol = '🔥'
                    bg_color = '#FF6B6B'
                else:
                    # Find best action from Q-values
                    best_action_idx = np.argmax(q_table[i, j])
                    best_action = Action(best_action_idx)
                    symbol = self.action_symbols[best_action]
                    bg_color = '#E8F5E8'
                
                html += f"<div style='width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border: 1px solid #ccc; background-color: {bg_color};'>{symbol}</div>"
            html += "</div>"
        
        html += "</div></div>"
        return html

print("Q-Table visualizer created successfully!")
print("✓ Q-value heatmap visualization")
print("✓ Policy extraction from Q-table")
print("✓ Real-time learning progress display")

Q-Table visualizer created successfully!
✓ Q-value heatmap visualization
✓ Policy extraction from Q-table
✓ Real-time learning progress display


---

# Part 3: Q-Learning Algorithm

## 3. Q-Learning Implementation

### Equations

**Q-Learning Update Rule:**
```
Q(s,a) ← Q(s,a) + α [r + γ max_a' Q(s',a') - Q(s,a)]
```

**Epsilon-Greedy Policy:**
```
π(s) = argmax_a Q(s,a)  with probability 1-ε
       random action    with probability ε
```

**Key Properties:**
- **Off-policy**: Learns optimal Q* regardless of behavior policy
- **Model-free**: No environment model required
- **Bootstrapping**: Updates estimates using other estimates

### How Q-Learning Works:
1. **Initialize** Q-table with zeros
2. **Choose action** using epsilon-greedy policy
3. **Take action** and observe reward and next state
4. **Update** Q-value using the Q-learning formula
5. **Repeat** until convergence or maximum episodes

Let's implement this step by step!

In [51]:
class QLearning:
    def __init__(self, grid_world, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.grid_world = grid_world
        self.alpha = alpha      # Learning rate
        self.gamma = gamma      # Discount factor
        self.epsilon = epsilon  # Exploration rate
        
        # Initialize Q-table: [height, width, num_actions]
        self.q_table = np.zeros((grid_world.height, grid_world.width, len(Action)))
        
        # Learning statistics
        self.episode_rewards = []
        self.episode_steps = []
        self.episode_count = 0
        
        # Create visualizer
        self.visualizer = QTableVisualizer(grid_world)
        
    def choose_action(self, state, use_epsilon=True):
        """Choose action using epsilon-greedy policy"""
        row, col = state
        
        if use_epsilon and random.random() < self.epsilon:
            # Exploration: choose random valid action
            valid_actions = []
            for action in Action:
                # Temporarily set position to check validity
                original_pos = self.grid_world.agent_pos.copy()
                self.grid_world.agent_pos = [row, col]
                if self.grid_world.is_valid_action(action):
                    valid_actions.append(action)
                self.grid_world.agent_pos = original_pos
            
            return random.choice(valid_actions) if valid_actions else Action.UP
        else:
            # Exploitation: choose action with highest Q-value
            action_idx = np.argmax(self.q_table[row, col])
            return Action(action_idx)
    
    def update_q_value(self, state, action, reward, next_state, done):
        """Update Q-value using Q-learning formula"""
        row, col = state
        next_row, next_col = next_state
        action_idx = action.value
        
        # Current Q-value
        current_q = self.q_table[row, col, action_idx]
        
        # Maximum Q-value for next state (off-policy)
        if done:
            max_next_q = 0  # Terminal state
        else:
            max_next_q = np.max(self.q_table[next_row, next_col])
        
        # Q-learning update
        target = reward + self.gamma * max_next_q
        td_error = target - current_q
        self.q_table[row, col, action_idx] = current_q + self.alpha * td_error
        
        return td_error
    
    def run_episode(self, max_steps=100):
        """Run one episode of Q-learning"""
        state = self.grid_world.reset()
        total_reward = 0
        steps = 0
        episode_transitions = []
        
        for step in range(max_steps):
            # Choose action
            action = self.choose_action(state)
            
            # Take action
            next_state, reward, done, _ = self.grid_world.step(action)
            
            # Update Q-value
            td_error = self.update_q_value(state, action, reward, next_state, done)
            
            # Store transition for visualization
            episode_transitions.append({
                'state': state,
                'action': action,
                'reward': reward,
                'next_state': next_state,
                'td_error': td_error,
                'done': done
            })
            
            total_reward += reward
            steps += 1
            state = next_state
            
            if done:
                break
        
        # Store episode statistics
        self.episode_rewards.append(total_reward)
        self.episode_steps.append(steps)
        self.episode_count += 1
        
        return episode_transitions, total_reward, steps
    
    def train(self, num_episodes=100, show_progress=True):
        """Train the Q-learning agent"""
        for episode in range(num_episodes):
            transitions, reward, steps = self.run_episode()
            
            if show_progress and (episode + 1) % 10 == 0:
                avg_reward = np.mean(self.episode_rewards[-10:])
                avg_steps = np.mean(self.episode_steps[-10:])
                print(f"Episode {episode + 1}: Avg Reward = {avg_reward:.2f}, Avg Steps = {avg_steps:.1f}")
    
    def get_policy(self):
        """Extract policy from Q-table"""
        policy = np.zeros((self.grid_world.height, self.grid_world.width), dtype=int)
        
        for i in range(self.grid_world.height):
            for j in range(self.grid_world.width):
                if self.grid_world.grid[i, j] != CellType.OBSTACLE.value:
                    best_action = np.argmax(self.q_table[i, j])
                    policy[i, j] = best_action
                else:
                    policy[i, j] = -1
        
        return policy
    
    def test_policy(self, max_steps=50):
        """Test the learned policy (no exploration)"""
        state = self.grid_world.reset()
        path = [state]
        total_reward = 0
        
        for step in range(max_steps):
            action = self.choose_action(state, use_epsilon=False)
            next_state, reward, done, _ = self.grid_world.step(action)
            
            path.append(next_state)
            total_reward += reward
            state = next_state
            
            if done:
                break
        
        return path, total_reward, len(path) - 1

print("Q-Learning algorithm implemented!")
print("✓ Epsilon-greedy exploration")
print("✓ Q-value updates")
print("✓ Policy extraction")
print("✓ Training and testing functionality")

Q-Learning algorithm implemented!
✓ Epsilon-greedy exploration
✓ Q-value updates
✓ Policy extraction
✓ Training and testing functionality


### Step-by-Step Q-Learning

Let's create an interactive version where you can see Q-Learning in action step by step!

In [52]:
class InteractiveQLearning:
    def __init__(self, grid_world):
        self.q_agent = QLearning(grid_world, alpha=0.1, gamma=0.9, epsilon=0.3)
        self.current_state = None
        self.episode_step = 0
        self.create_widgets()
        
    def create_widgets(self):
        """Create interactive controls"""
        self.step_button = widgets.Button(description='➡️ Next Step')
        self.episode_button = widgets.Button(description='🎬 Full Episode')
        self.train_button = widgets.Button(description='🏋️ Train 10 Episodes')
        self.reset_button = widgets.Button(description='🔄 Reset')
        self.test_button = widgets.Button(description='🧪 Test Policy')
        
        # Parameter controls
        self.alpha_slider = widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, description='Learning Rate (α):')
        self.epsilon_slider = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.01, description='Exploration (ε):')
        
        self.step_button.on_click(lambda b: self.single_step())
        self.episode_button.on_click(lambda b: self.full_episode())
        self.train_button.on_click(lambda b: self.train_episodes())
        self.reset_button.on_click(lambda b: self.reset())
        self.test_button.on_click(lambda b: self.test_policy())
        
        self.alpha_slider.observe(self.update_parameters, names='value')
        self.epsilon_slider.observe(self.update_parameters, names='value')
        
        self.output = widgets.Output()
        self.display_area = widgets.Output()
        
        self.controls = widgets.VBox([
            widgets.HTML("<h4>🎛️ Controls</h4>"),
            widgets.HBox([self.step_button, self.episode_button, self.train_button]),
            widgets.HBox([self.reset_button, self.test_button]),
            widgets.HTML("<h4>⚙️ Parameters</h4>"),
            self.alpha_slider,
            self.epsilon_slider
        ])
        
    def update_parameters(self, change):
        """Update algorithm parameters"""
        self.q_agent.alpha = self.alpha_slider.value
        self.q_agent.epsilon = self.epsilon_slider.value
        
    def single_step(self):
        """Execute one step of Q-learning"""
        with self.output:
            clear_output(wait=True)
            
            if self.current_state is None:
                # Start new episode
                self.current_state = self.q_agent.grid_world.reset()
                self.episode_step = 0
                print(f"🚀 Starting Episode {self.q_agent.episode_count + 1}")
                print(f"🎯 Initial state: {self.current_state}")
                self.update_display()
                return
            
            # Choose action
            action = self.q_agent.choose_action(self.current_state)
            exploration = "🎲 Exploration" if random.random() < self.q_agent.epsilon else "🎯 Exploitation"
            
            # Take action
            next_state, reward, done, _ = self.q_agent.grid_world.step(action)
            
            # Store old Q-value
            old_q = self.q_agent.q_table[self.current_state[0], self.current_state[1], action.value]
            
            # Update Q-value
            td_error = self.q_agent.update_q_value(self.current_state, action, reward, next_state, done)
            
            # Display step information
            new_q = self.q_agent.q_table[self.current_state[0], self.current_state[1], action.value]
            print(f"Step {self.episode_step + 1}:")
            print(f"  State: {self.current_state} → Action: {action.name} ({exploration})")
            print(f"  Next State: {next_state}, Reward: {reward:+.1f}")
            print(f"  Q-value: {old_q:.3f} → {new_q:.3f} (TD Error: {td_error:+.3f})")
            
            self.episode_step += 1
            self.current_state = next_state
            
            if done:
                print(f"\n✅ Episode {self.q_agent.episode_count + 1} completed in {self.episode_step} steps!")
                self.q_agent.episode_count += 1
                self.current_state = None
            
            self.update_display()
    
    def full_episode(self):
        """Run a complete episode"""
        with self.output:
            clear_output(wait=True)
            
            transitions, reward, steps = self.q_agent.run_episode()
            
            print(f"🎬 Episode {self.q_agent.episode_count} completed:")
            print(f"  Steps: {steps}")
            print(f"  Total Reward: {reward:+.1f}")
            print(f"  Average Reward per Step: {reward/steps:+.2f}")
            
            self.current_state = None
            self.update_display()
    
    def train_episodes(self):
        """Train for multiple episodes"""
        with self.output:
            clear_output(wait=True)
            
            print("🏋️ Training for 10 episodes...")
            start_episode = self.q_agent.episode_count
            
            for i in range(10):
                transitions, reward, steps = self.q_agent.run_episode()
                if (i + 1) % 5 == 0:
                    print(f"  Episode {start_episode + i + 1}: {steps} steps, {reward:+.1f} reward")
            
            avg_reward = np.mean(self.q_agent.episode_rewards[-10:])
            avg_steps = np.mean(self.q_agent.episode_steps[-10:])
            
            print(f"\n📊 Training Summary:")
            print(f"  Episodes: {start_episode + 1} → {self.q_agent.episode_count}")
            print(f"  Average Reward (last 10): {avg_reward:+.2f}")
            print(f"  Average Steps (last 10): {avg_steps:.1f}")
            
            self.current_state = None
            self.update_display()
    
    def test_policy(self):
        """Test the learned policy"""
        with self.output:
            clear_output(wait=True)
            
            print("🧪 Testing learned policy (no exploration)...")
            path, reward, steps = self.q_agent.test_policy()
            
            print(f"🏆 Policy Test Results:")
            print(f"  Path: {' → '.join(map(str, path))}")
            print(f"  Steps: {steps}")
            print(f"  Total Reward: {reward:+.1f}")
            
            if reward > 5:  # Assuming goal reward is 10, minus some step penalties
                print("✅ Good policy! Successfully reached the goal.")
            else:
                print("📚 Policy needs more training.")
    
    def reset(self):
        """Reset the algorithm"""
        with self.output:
            clear_output(wait=True)
            
            self.q_agent = QLearning(self.q_agent.grid_world, 
                                   alpha=self.alpha_slider.value, 
                                   gamma=0.9, 
                                   epsilon=self.epsilon_slider.value)
            self.current_state = None
            self.episode_step = 0
            
            print("🔄 Q-Learning algorithm reset!")
            print("📊 Q-table initialized to zeros")
            print("➡️ Click 'Next Step' to start learning")
            
            self.update_display()
    
    def update_display(self):
        """Update the visualization"""
        with self.display_area:
            clear_output(wait=True)
            
            # Create visualizations
            grid_html = self.q_agent.visualizer.create_grid_html()
            q_table_html = self.q_agent.visualizer.create_q_table_html(self.q_agent.q_table)
            policy_html = self.q_agent.visualizer.create_policy_from_q_html(self.q_agent.q_table)
            
            grid_widget = widgets.HTML(value=grid_html)
            q_table_widget = widgets.HTML(value=q_table_html)
            policy_widget = widgets.HTML(value=policy_html)
            
            # Statistics
            stats_html = f"""
            <div style='padding: 10px; background-color: #f0f0f0; margin: 10px 0; border-radius: 5px;'>
                <h4>📊 Learning Statistics</h4>
                <p><strong>Episodes:</strong> {self.q_agent.episode_count}</p>
                <p><strong>Learning Rate (α):</strong> {self.q_agent.alpha}</p>
                <p><strong>Exploration (ε):</strong> {self.q_agent.epsilon}</p>
            </div>
            """
            
            display(widgets.VBox([
                widgets.HTML(f"<h4>🤖 Current Environment (Episode {self.q_agent.episode_count + 1})</h4>"),
                widgets.HBox([grid_widget, widgets.HTML(stats_html)]),
                widgets.HTML("<h4>📊 Q-Table and Policy</h4>"),
                widgets.VBox([
                    q_table_widget,
                    policy_widget
                ])
            ]))
    
    def start_interactive(self):
        """Start the interactive interface"""
        self.reset()
        return widgets.VBox([
            widgets.HTML("<h3>🧠 Interactive Q-Learning</h3>"),
            widgets.HTML("<p><strong>Q-Learning</strong> learns optimal action-values through trial and error!</p>"),
            self.controls,
            self.output,
            self.display_area
        ])

# Create interactive Q-Learning
interactive_q = InteractiveQLearning(GridWorld(width=5, height=5))
print("🎮 Interactive Q-Learning ready!")
print("Adjust parameters and watch the algorithm learn step by step.")

# Display the interface
display(interactive_q.start_interactive())

🎮 Interactive Q-Learning ready!
Adjust parameters and watch the algorithm learn step by step.


---

# Part 4: SARSA Algorithm

## 4. SARSA Implementation

### Equations

**SARSA Update Rule:**
```
Q(s,a) ← Q(s,a) + α [r + γ Q(s',a') - Q(s,a)]
```

**Key Differences from Q-Learning:**
- **On-policy**: Learns Q-values for the policy being followed
- **Uses actual next action**: Updates with Q(s',a') where a' is the action actually taken
- **More conservative**: Accounts for exploration in the learning process

### How SARSA Works:
1. **Initialize** Q-table with zeros
2. **Choose action** using epsilon-greedy policy
3. **Take action** and observe reward and next state
4. **Choose next action** using same policy
5. **Update** Q-value using the SARSA formula with actual next action
6. **Repeat** until convergence or maximum episodes

The key insight: SARSA learns the value of the policy it's actually following (including exploration), while Q-Learning learns the optimal policy regardless of exploration.

Let's implement SARSA!

In [53]:
class SARSA:
    def __init__(self, grid_world, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.grid_world = grid_world
        self.alpha = alpha      # Learning rate
        self.gamma = gamma      # Discount factor
        self.epsilon = epsilon  # Exploration rate
        
        # Initialize Q-table: [height, width, num_actions]
        self.q_table = np.zeros((grid_world.height, grid_world.width, len(Action)))
        
        # Learning statistics
        self.episode_rewards = []
        self.episode_steps = []
        self.episode_count = 0
        
        # Create visualizer
        self.visualizer = QTableVisualizer(grid_world)
        
    def choose_action(self, state, use_epsilon=True):
        """Choose action using epsilon-greedy policy"""
        row, col = state
        
        if use_epsilon and random.random() < self.epsilon:
            # Exploration: choose random valid action
            valid_actions = []
            for action in Action:
                # Temporarily set position to check validity
                original_pos = self.grid_world.agent_pos.copy()
                self.grid_world.agent_pos = [row, col]
                if self.grid_world.is_valid_action(action):
                    valid_actions.append(action)
                self.grid_world.agent_pos = original_pos
            
            return random.choice(valid_actions) if valid_actions else Action.UP
        else:
            # Exploitation: choose action with highest Q-value
            action_idx = np.argmax(self.q_table[row, col])
            return Action(action_idx)
    
    def update_q_value(self, state, action, reward, next_state, next_action, done):
        """Update Q-value using SARSA formula"""
        row, col = state
        next_row, next_col = next_state
        action_idx = action.value
        next_action_idx = next_action.value if next_action else 0
        
        # Current Q-value
        current_q = self.q_table[row, col, action_idx]
        
        # Q-value for next state-action pair (on-policy)
        if done:
            next_q = 0  # Terminal state
        else:
            next_q = self.q_table[next_row, next_col, next_action_idx]
        
        # SARSA update
        target = reward + self.gamma * next_q
        td_error = target - current_q
        self.q_table[row, col, action_idx] = current_q + self.alpha * td_error
        
        return td_error
    
    def run_episode(self, max_steps=100):
        """Run one episode of SARSA"""
        state = self.grid_world.reset()
        action = self.choose_action(state)  # Choose initial action
        
        total_reward = 0
        steps = 0
        episode_transitions = []
        
        for step in range(max_steps):
            # Take action
            next_state, reward, done, _ = self.grid_world.step(action)
            
            # Choose next action (for SARSA update)
            if not done:
                next_action = self.choose_action(next_state)
            else:
                next_action = None
            
            # Update Q-value using SARSA
            td_error = self.update_q_value(state, action, reward, next_state, next_action, done)
            
            # Store transition for visualization
            episode_transitions.append({
                'state': state,
                'action': action,
                'reward': reward,
                'next_state': next_state,
                'next_action': next_action,
                'td_error': td_error,
                'done': done
            })
            
            total_reward += reward
            steps += 1
            
            if done:
                break
                
            # Move to next state-action pair
            state = next_state
            action = next_action
        
        # Store episode statistics
        self.episode_rewards.append(total_reward)
        self.episode_steps.append(steps)
        self.episode_count += 1
        
        return episode_transitions, total_reward, steps
    
    def train(self, num_episodes=100, show_progress=True):
        """Train the SARSA agent"""
        for episode in range(num_episodes):
            transitions, reward, steps = self.run_episode()
            
            if show_progress and (episode + 1) % 10 == 0:
                avg_reward = np.mean(self.episode_rewards[-10:])
                avg_steps = np.mean(self.episode_steps[-10:])
                print(f"Episode {episode + 1}: Avg Reward = {avg_reward:.2f}, Avg Steps = {avg_steps:.1f}")
    
    def get_policy(self):
        """Extract policy from Q-table"""
        policy = np.zeros((self.grid_world.height, self.grid_world.width), dtype=int)
        
        for i in range(self.grid_world.height):
            for j in range(self.grid_world.width):
                if self.grid_world.grid[i, j] != CellType.OBSTACLE.value:
                    best_action = np.argmax(self.q_table[i, j])
                    policy[i, j] = best_action
                else:
                    policy[i, j] = -1
        
        return policy
    
    def test_policy(self, max_steps=50):
        """Test the learned policy (no exploration)"""
        state = self.grid_world.reset()
        path = [state]
        total_reward = 0
        
        for step in range(max_steps):
            action = self.choose_action(state, use_epsilon=False)
            next_state, reward, done, _ = self.grid_world.step(action)
            
            path.append(next_state)
            total_reward += reward
            state = next_state
            
            if done:
                break
        
        return path, total_reward, len(path) - 1

print("SARSA algorithm implemented!")
print("✓ On-policy learning")
print("✓ State-Action-Reward-State-Action updates")
print("✓ Conservative exploration handling")
print("✓ Training and testing functionality")

SARSA algorithm implemented!
✓ On-policy learning
✓ State-Action-Reward-State-Action updates
✓ Conservative exploration handling
✓ Training and testing functionality


### Step-by-Step SARSA

Now let's create an interactive SARSA implementation to see how it differs from Q-Learning!

In [54]:
class InteractiveSARSA:
    def __init__(self, grid_world):
        self.sarsa_agent = SARSA(grid_world, alpha=0.1, gamma=0.9, epsilon=0.3)
        self.current_state = None
        self.current_action = None
        self.episode_step = 0
        self.create_widgets()
        
    def create_widgets(self):
        """Create interactive controls"""
        self.step_button = widgets.Button(description='➡️ Next Step')
        self.episode_button = widgets.Button(description='🎬 Full Episode')
        self.train_button = widgets.Button(description='🏋️ Train 10 Episodes')
        self.reset_button = widgets.Button(description='🔄 Reset')
        self.test_button = widgets.Button(description='🧪 Test Policy')
        
        # Parameter controls
        self.alpha_slider = widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, description='Learning Rate (α):')
        self.epsilon_slider = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.01, description='Exploration (ε):')
        
        self.step_button.on_click(lambda b: self.single_step())
        self.episode_button.on_click(lambda b: self.full_episode())
        self.train_button.on_click(lambda b: self.train_episodes())
        self.reset_button.on_click(lambda b: self.reset())
        self.test_button.on_click(lambda b: self.test_policy())
        
        self.alpha_slider.observe(self.update_parameters, names='value')
        self.epsilon_slider.observe(self.update_parameters, names='value')
        
        self.output = widgets.Output()
        self.display_area = widgets.Output()
        
        self.controls = widgets.VBox([
            widgets.HTML("<h4>🎛️ Controls</h4>"),
            widgets.HBox([self.step_button, self.episode_button, self.train_button]),
            widgets.HBox([self.reset_button, self.test_button]),
            widgets.HTML("<h4>⚙️ Parameters</h4>"),
            self.alpha_slider,
            self.epsilon_slider
        ])
        
    def update_parameters(self, change):
        """Update algorithm parameters"""
        self.sarsa_agent.alpha = self.alpha_slider.value
        self.sarsa_agent.epsilon = self.epsilon_slider.value
        
    def single_step(self):
        """Execute one step of SARSA"""
        with self.output:
            clear_output(wait=True)
            
            if self.current_state is None:
                # Start new episode
                self.current_state = self.sarsa_agent.grid_world.reset()
                self.current_action = self.sarsa_agent.choose_action(self.current_state)
                self.episode_step = 0
                print(f"🚀 Starting Episode {self.sarsa_agent.episode_count + 1}")
                print(f"🎯 Initial state: {self.current_state}")
                print(f"🎲 Chosen action: {self.current_action.name}")
                self.update_display()
                return
            
            # Take action (we already have current_action from previous step or start)
            next_state, reward, done, _ = self.sarsa_agent.grid_world.step(self.current_action)
            
            # Choose next action (needed for SARSA update)
            if not done:
                next_action = self.sarsa_agent.choose_action(next_state)
                exploration = "🎲 Exploration" if random.random() < self.sarsa_agent.epsilon else "🎯 Exploitation"
            else:
                next_action = None
                exploration = "N/A (Terminal)"
            
            # Store old Q-value
            old_q = self.sarsa_agent.q_table[self.current_state[0], self.current_state[1], self.current_action.value]
            
            # Update Q-value using SARSA
            td_error = self.sarsa_agent.update_q_value(self.current_state, self.current_action, reward, next_state, next_action, done)
            
            # Display step information
            new_q = self.sarsa_agent.q_table[self.current_state[0], self.current_state[1], self.current_action.value]
            print(f"Step {self.episode_step + 1}:")
            print(f"  State: {self.current_state} → Action: {self.current_action.name}")
            print(f"  Next State: {next_state}, Reward: {reward:+.1f}")
            if next_action:
                print(f"  Next Action: {next_action.name} ({exploration})")
            print(f"  Q-value: {old_q:.3f} → {new_q:.3f} (TD Error: {td_error:+.3f})")
            print(f"  💡 SARSA uses Q(s',a') = {self.sarsa_agent.q_table[next_state[0], next_state[1], next_action.value] if next_action else 0:.3f}")
            
            self.episode_step += 1
            self.current_state = next_state
            self.current_action = next_action
            
            if done:
                print(f"\n✅ Episode {self.sarsa_agent.episode_count + 1} completed in {self.episode_step} steps!")
                self.sarsa_agent.episode_count += 1
                self.current_state = None
                self.current_action = None
            
            self.update_display()
    
    def full_episode(self):
        """Run a complete episode"""
        with self.output:
            clear_output(wait=True)
            
            transitions, reward, steps = self.sarsa_agent.run_episode()
            
            print(f"🎬 Episode {self.sarsa_agent.episode_count} completed:")
            print(f"  Steps: {steps}")
            print(f"  Total Reward: {reward:+.1f}")
            print(f"  Average Reward per Step: {reward/steps:+.2f}")
            
            # Show some key transitions
            print(f"\n📋 Key Transitions:")
            for i, transition in enumerate(transitions[:3]):  # Show first 3
                s, a, r, s_next, a_next = transition['state'], transition['action'], transition['reward'], transition['next_state'], transition['next_action']
                print(f"  {i+1}. {s} --{a.name}→ {s_next} (R: {r:+.1f}, Next: {a_next.name if a_next else 'None'})")
            
            self.current_state = None
            self.current_action = None
            self.update_display()
    
    def train_episodes(self):
        """Train for multiple episodes"""
        with self.output:
            clear_output(wait=True)
            
            print("🏋️ Training for 10 episodes...")
            start_episode = self.sarsa_agent.episode_count
            
            for i in range(10):
                transitions, reward, steps = self.sarsa_agent.run_episode()
                if (i + 1) % 5 == 0:
                    print(f"  Episode {start_episode + i + 1}: {steps} steps, {reward:+.1f} reward")
            
            avg_reward = np.mean(self.sarsa_agent.episode_rewards[-10:])
            avg_steps = np.mean(self.sarsa_agent.episode_steps[-10:])
            
            print(f"\n📊 Training Summary:")
            print(f"  Episodes: {start_episode + 1} → {self.sarsa_agent.episode_count}")
            print(f"  Average Reward (last 10): {avg_reward:+.2f}")
            print(f"  Average Steps (last 10): {avg_steps:.1f}")
            
            self.current_state = None
            self.current_action = None
            self.update_display()
    
    def test_policy(self):
        """Test the learned policy"""
        with self.output:
            clear_output(wait=True)
            
            print("🧪 Testing learned policy (no exploration)...")
            path, reward, steps = self.sarsa_agent.test_policy()
            
            print(f"🏆 Policy Test Results:")
            print(f"  Path: {' → '.join(map(str, path))}")
            print(f"  Steps: {steps}")
            print(f"  Total Reward: {reward:+.1f}")
            
            if reward > 5:  # Assuming goal reward is 10, minus some step penalties
                print("✅ Good policy! Successfully reached the goal.")
            else:
                print("📚 Policy needs more training.")
    
    def reset(self):
        """Reset the algorithm"""
        with self.output:
            clear_output(wait=True)
            
            self.sarsa_agent = SARSA(self.sarsa_agent.grid_world, 
                                   alpha=self.alpha_slider.value, 
                                   gamma=0.9, 
                                   epsilon=self.epsilon_slider.value)
            self.current_state = None
            self.current_action = None
            self.episode_step = 0
            
            print("🔄 SARSA algorithm reset!")
            print("📊 Q-table initialized to zeros")
            print("➡️ Click 'Next Step' to start learning")
            
            self.update_display()
    
    def update_display(self):
        """Update the visualization"""
        with self.display_area:
            clear_output(wait=True)
            
            # Create visualizations
            grid_html = self.sarsa_agent.visualizer.create_grid_html()
            q_table_html = self.sarsa_agent.visualizer.create_q_table_html(self.sarsa_agent.q_table, "SARSA Q-Table")
            policy_html = self.sarsa_agent.visualizer.create_policy_from_q_html(self.sarsa_agent.q_table, "SARSA Policy")
            
            grid_widget = widgets.HTML(value=grid_html)
            q_table_widget = widgets.HTML(value=q_table_html)
            policy_widget = widgets.HTML(value=policy_html)
            
            # Statistics
            stats_html = f"""
            <div style='padding: 10px; background-color: #f0f0f0; margin: 10px 0; border-radius: 5px;'>
                <h4>📊 Learning Statistics</h4>
                <p><strong>Episodes:</strong> {self.sarsa_agent.episode_count}</p>
                <p><strong>Learning Rate (α):</strong> {self.sarsa_agent.alpha}</p>
                <p><strong>Exploration (ε):</strong> {self.sarsa_agent.epsilon}</p>
                <p><strong>Current State:</strong> {self.current_state}</p>
                <p><strong>Next Action:</strong> {self.current_action.name if self.current_action else 'None'}</p>
            </div>
            """
            
            display(widgets.VBox([
                widgets.HTML(f"<h4>🤖 SARSA Learning (Episode {self.sarsa_agent.episode_count + 1})</h4>"),
                widgets.HBox([grid_widget, widgets.HTML(stats_html)]),
                widgets.HTML("<h4>📊 Q-Table and Policy</h4>"),
                widgets.VBox([
                    q_table_widget,
                    policy_widget
                ])
            ]))
    
    def start_interactive(self):
        """Start the interactive interface"""
        self.reset()
        return widgets.VBox([
            widgets.HTML("<h3>🧠 Interactive SARSA</h3>"),
            widgets.HTML("<p><strong>SARSA</strong> learns action-values for the policy being followed (on-policy)!</p>"),
            widgets.HTML("<p>🔍 Key difference: Uses Q(s',a') where a' is the action actually chosen next.</p>"),
            self.controls,
            self.output,
            self.display_area
        ])

# Create interactive SARSA
interactive_sarsa = InteractiveSARSA(GridWorld(width=5, height=5))
print("🎮 Interactive SARSA ready!")
print("Compare how SARSA differs from Q-Learning in its updates.")

# Display the interface
display(interactive_sarsa.start_interactive())

🎮 Interactive SARSA ready!
Compare how SARSA differs from Q-Learning in its updates.


## 🎯 Learning Summary

After exploring Q-Learning and SARSA, consider these key insights:

### 🔍 **Key Differences:**

| Aspect | Q-Learning | SARSA |
|--------|------------|-------|
| **Policy Type** | Off-policy | On-policy |
| **Update Rule** | Uses max Q(s',a') | Uses actual Q(s',a') |
| **Behavior** | More aggressive | More conservative |
| **Exploration Impact** | Ignores exploration in updates | Considers exploration in updates |
| **Convergence** | To optimal policy | To policy being followed |

### 🧠 **Conceptual Understanding:**

1. **Q-Learning**: "What's the best I could do from here?" (regardless of what I actually do)
2. **SARSA**: "What will I actually get if I keep following my current strategy?"

### 🚀 **When to Use Which:**

- **Q-Learning**: When you want the optimal policy and can tolerate aggressive exploration
- **SARSA**: When safety matters and you want a policy that accounts for exploration risks

### 🎮 **Exploration Strategies:**

Both algorithms use **epsilon-greedy** exploration:
- **High ε**: More exploration, slower convergence, better final policy
- **Low ε**: Less exploration, faster convergence, might miss optimal policy